In [ ]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GroupShuffleSplit,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBClassifier
import joblib
import os
from pathlib import Path

# Find the project root directory
ROOT = Path.cwd()

if not (ROOT / "datasets").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "datasets"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("All libraries imported!")
print("Project root:", ROOT)
print("Dataset folder:", DATA_DIR)
print("Model folder:", MODEL_DIR)


In [ ]:
df = pd.read_csv(DATA_DIR / "Crop_recommendation.csv")

print("Crop dataset loaded!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

print("\nDuplicate rows:", df.duplicated().sum())
print("Missing values:", int(df.isna().sum().sum()))
print("Rows per class:")
print(df["label"].value_counts().sort_index())

print(
    "\nWarning: every class has exactly 100 rows, and the temperature, "
    "humidity, pH and rainfall values are unique per row. This looks like "
    "a synthetic benchmark rather than field data."
)


In [ ]:
X = df[['N', 'P', 'K', 'temperature',
        'humidity', 'ph', 'rainfall']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

crop_model = RandomForestClassifier(
    n_estimators=100, random_state=42
)
crop_model.fit(X_train, y_train)

y_pred = crop_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Crop Model Accuracy:", round(acc * 100, 2), "%")
print(classification_report(y_test, y_pred))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    crop_model, X, y, cv=cv, scoring="accuracy"
)
print("5-fold CV accuracy:", round(cv_scores.mean() * 100, 2), "%")
print("CV standard deviation:", round(cv_scores.std() * 100, 2), "%")

importance = pd.Series(
    crop_model.feature_importances_,
    index=X.columns,
).sort_values(ascending=False)
print("\nFeature importances:")
print(importance.round(4))

print(
    "\nWarning: this benchmark is perfectly balanced and the environmental "
    "values are unique per row. The high score does not prove field accuracy."
)

joblib.dump(crop_model, MODEL_DIR / "crop_model.pkl")
print("Crop model saved!")

In [ ]:
fert_check = pd.read_csv(DATA_DIR / "Fertilizer Prediction.csv")
fert_check.columns = fert_check.columns.str.strip()

print("Original dataset shape:", fert_check.shape)
print("\nOriginal fertilizer counts:")
print(fert_check["Fertilizer Name"].value_counts())

print("\nExact duplicate rows:", fert_check.duplicated().sum())

date_label = "10/26/2026"
date_rows = (
    fert_check["Fertilizer Name"].astype(str).str.strip() == date_label
)
print(f"Rows with date-like label '{date_label}':", int(date_rows.sum()))
print(
    "Unique rows with that label:",
    fert_check.loc[date_rows].drop_duplicates().shape[0],
)
print(
    "The label is almost certainly an Excel conversion of the fertilizer "
    "grade '10-26-26', so it is excluded instead of being treated as a class."
)

fert_clean = fert_check[~date_rows].copy()
fert_clean = fert_clean.drop_duplicates().copy()

print("\nCleaned dataset shape:", fert_clean.shape)
print("\nCleaned fertilizer counts:")
print(fert_clean["Fertilizer Name"].value_counts())
print("\nRemaining duplicates:", fert_clean.duplicated().sum())

npk_cols = ["Nitrogen", "Phosphorous", "Potassium"]
npk_groups = fert_clean.groupby(npk_cols)["Fertilizer Name"].nunique()
print("\nDistinct NPK combinations:", len(npk_groups))
print(
    "NPK combinations with more than one fertilizer label:",
    int((npk_groups > 1).sum()),
)
print(
    "Warning: the fertilizer label is almost fully determined by the N, P "
    "and K values. The model learns a grade-matching rule, not an "
    "independent agronomic recommendation."
)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

train_df = fert_clean.copy()

# Create encoders
le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

# Encode input categories
train_df["Soil Type"] = le_soil.fit_transform(
    train_df["Soil Type"].astype(str)
)

train_df["Crop Type"] = le_crop.fit_transform(
    train_df["Crop Type"].astype(str)
)

# Encode the target
train_df["Fertilizer Name"] = le_fert.fit_transform(
    train_df["Fertilizer Name"].astype(str)
)

# Keep the feature order consistent with the backend
fert_features = [
    "Temparature",
    "Humidity",
    "Moisture",
    "Soil Type",
    "Crop Type",
    "Nitrogen",
    "Potassium",
    "Phosphorous"
]

X2 = train_df[fert_features]
y2 = train_df["Fertilizer Name"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42,
    stratify=y2
)

print("Training rows:", len(X2_train))
print("Testing rows:", len(X2_test))
print("Number of fertilizer classes:", len(le_fert.classes_))
print("Fertilizer classes:", list(le_fert.classes_))

In [ ]:
fert_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="mlogloss"
)

fert_model.fit(X2_train, y2_train)

print("Fertilizer model training completed!")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y2_train_pred = fert_model.predict(X2_train)
y2_test_pred = fert_model.predict(X2_test)

print("Training accuracy:",
      round(accuracy_score(y2_train, y2_train_pred) * 100, 2), "%")

print("Testing accuracy:",
      round(accuracy_score(y2_test, y2_test_pred) * 100, 2), "%")

print("\nClassification report:")
print(
    classification_report(
        y2_test,
        y2_test_pred,
        labels=range(len(le_fert.classes_)),
        target_names=le_fert.classes_,
        zero_division=0
    )
)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from xgboost import XGBClassifier

X_raw = fert_clean.drop(columns=["Fertilizer Name"]).copy()
y_raw = fert_clean["Fertilizer Name"].astype(str)

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y_raw)

categorical_cols = ["Soil Type", "Crop Type"]
numeric_cols = [c for c in X_raw.columns if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ("categories", OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1
    ), categorical_cols),
    ("numbers", "passthrough", numeric_cols)
])

cv_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    cv_model, X_raw, y_encoded, cv=cv, scoring="accuracy"
)

print("Fold accuracies:", (scores * 100).round(2))
print("Mean CV accuracy:", round(scores.mean() * 100, 2), "%")
print("Standard deviation:", round(scores.std() * 100, 2), "%")

print("\nRows used for cross-validation:", len(X_raw))
print("Target classes:", sorted(y_raw.unique()))
print(
    "\nWarning: the cleaned fertilizer dataset has only 92 rows. A 19-row "
    "hold-out and 5-fold CV are sanity checks, not evidence of field accuracy."
)


In [ ]:
import joblib

MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(fert_model, MODEL_DIR / "fert_model.pkl")
joblib.dump(le_soil, MODEL_DIR / "le_soil.pkl")
joblib.dump(le_crop, MODEL_DIR / "le_crop.pkl")
joblib.dump(le_fert, MODEL_DIR / "le_fert.pkl")

print("Fertilizer model and encoders saved successfully!")
print("Saved to:", MODEL_DIR)

## Yield data: duplicates, merge artifacts, and honest evaluation

The raw yield file contains many exact duplicate rows and many repeated
`Area + Item + Year` keys. The merge that created it also introduces
many-to-many matches, so several weather values can appear for the same
yield record. The evaluation below therefore uses a deduplicated dataset,
a leave-series-out split, and a time split, in addition to the usual
random split.

In [ ]:
yield2 = pd.read_csv(DATA_DIR / "yield_df.csv")
yield2.columns = yield2.columns.str.strip()
yield2 = yield2.drop(columns=["Unnamed: 0"], errors="ignore")
yield2 = yield2.dropna().copy()

print("Yield dataset shape:", yield2.shape)
print("Exact duplicate rows:", yield2.duplicated().sum())

yield2 = yield2.drop_duplicates().copy()
print("Shape after removing exact duplicates:", yield2.shape)

key = ["Area", "Item", "Year"]
print(
    "Unique Area + Item + Year keys:",
    yield2[key].drop_duplicates().shape[0],
)
print(
    "Duplicate Area + Item + Year keys:",
    yield2.duplicated(subset=key).sum(),
)

india_rice = yield2[
    (yield2["Area"] == "India")
    & (yield2["Item"] == "Rice, paddy")
    & (yield2["Year"] == 2013)
]
print("\nIndia / Rice, paddy / 2013 rows:", len(india_rice))
print("Distinct yield values:", india_rice["hg/ha_yield"].nunique())
print("Distinct avg_temp values:", india_rice["avg_temp"].nunique())
print(
    "The many-to-many merge is visible when one yield value is paired with "
    "several different temperature values for the same area, crop and year."
)

print(
    "\nYear range:",
    int(yield2["Year"].min()),
    "to",
    int(yield2["Year"].max()),
)
print("Warning: these are country/area-year aggregates, not farm-level data.")

In [ ]:
df_yield = yield2.copy()

le_area = LabelEncoder()
le_item = LabelEncoder()

df_yield["Area"] = le_area.fit_transform(df_yield["Area"].astype(str))
df_yield["Item"] = le_item.fit_transform(df_yield["Item"].astype(str))

X3 = df_yield[
    [
        "Area",
        "Item",
        "Year",
        "average_rain_fall_mm_per_year",
        "pesticides_tonnes",
        "avg_temp",
    ]
]
y3 = df_yield["hg/ha_yield"]

print("Rows used for evaluation:", len(df_yield))
print(
    "Unique Area + Item series:",
    df_yield[["Area", "Item"]].drop_duplicates().shape[0],
)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

random_model = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)
random_model.fit(X3_train, y3_train)
random_pred = random_model.predict(X3_test)

print(
    "Random-split R2:",
    round(r2_score(y3_test, random_pred), 4),
)
print(
    "Random-split RMSE:",
    round(mean_squared_error(y3_test, random_pred) ** 0.5, 2),
)

groups = df_yield["Area"].astype(str) + " | " + df_yield["Item"].astype(str)
group_split = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)
group_train, group_test = next(
    group_split.split(X3, y3, groups=groups)
)

group_model = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)
group_model.fit(X3.iloc[group_train], y3.iloc[group_train])
group_pred = group_model.predict(X3.iloc[group_test])
print(
    "Unseen Area + Item series R2:",
    round(r2_score(y3.iloc[group_test], group_pred), 4),
)

train_mask = df_yield["Year"] <= 2010
test_mask = df_yield["Year"] > 2010

time_model = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)
time_model.fit(X3[train_mask], y3[train_mask])
time_pred = time_model.predict(X3[test_mask])
print(
    "Time-split R2 (train <= 2010, test > 2010):",
    round(r2_score(y3[test_mask], time_pred), 4),
)

yield_model = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1
)
yield_model.fit(X3, y3)

print(
    "\nWarning: the model cannot extrapolate beyond the observed years "
    "(1990-2013). Predictions for later years are not genuine forecasts."
)
print(
    "Warning: random-split R2 is inflated by duplicate series and merge "
    "artifacts. Use the leave-series-out and time-split scores instead."
)

joblib.dump(yield_model, MODEL_DIR / "yield_model.pkl")
joblib.dump(le_area, MODEL_DIR / "le_area.pkl")
joblib.dump(le_item, MODEL_DIR / "le_item.pkl")

print("Yield model and encoders saved successfully!")
print("Saved to:", MODEL_DIR)

In [ ]:
for file in sorted(MODEL_DIR.glob("*.pkl")):
    print(file.name)

In [ ]:
!pip install matplotlib

In [ ]:
import joblib

print("=== Saved model files ===")
for file in sorted(MODEL_DIR.glob("*.pkl")):
    print(file.name)

print("\n=== Fertilizer encoder classes ===")
print("Soil types:", le_soil.classes_)
print("Crop types:", le_crop.classes_)
print("Fertilizer types:", le_fert.classes_)

print("\n=== Yield encoder classes ===")
print("Areas:", le_area.classes_)
print("Yield crop types:", le_item.classes_)